# 04 — Candidate features và residual listwise reranker

Pipeline Coveo v1 — mọi output được version hóa và không ghi đè.

In [ ]:
%pip install -q -e ".[coveo]"

In [ ]:
from pathlib import Path
import os, json, yaml

def find_repo():
    here = Path.cwd().resolve()
    for root in (here, *here.parents):
        if (root / 'pyproject.toml').exists(): return root
    raise FileNotFoundError('Không tìm thấy pyproject.toml')

REPO = find_repo()
os.chdir(REPO)
cfg = yaml.safe_load((REPO / 'configs/coveo.yaml').read_text(encoding='utf-8'))
PROFILE = os.getenv('COVEO_PROFILE', cfg['project']['profile'])
print('repo=', REPO, 'profile=', PROFILE)

In [ ]:
from datn.recommenders.coveo.pipeline import RerankerTrainConfig, generate_candidates, train_reranker
rr = cfg['reranker']
params = RerankerTrainConfig(candidate_k=rr['candidate_k'], retrieval_k=rr['retrieval_k'], popularity_k=rr['popularity_k'],
    max_sessions_per_split=rr[f'max_sessions_per_split_{PROFILE}'], batch_size=rr['batch_size'], epochs=rr[f'epochs_{PROFILE}'],
    lr=rr['lr'], weight_decay=rr['weight_decay'], patience=rr['patience'], seed=cfg['project']['seed'], device=rr['device'])
processed = REPO / cfg['paths']['processed_dir']; embeddings = REPO / cfg['paths']['embeddings_dir']
retrieval_ckpt = REPO / cfg['paths']['retrieval_dir'] / 'best_retrieval.pt'
reranker_dir = REPO / cfg['paths']['reranker_dir']; candidate_dir = reranker_dir / 'candidates'
reports = {}
for split in ('train', 'valid', 'test'):
    reports[split] = generate_candidates(split, processed, embeddings, retrieval_ckpt, candidate_dir / f'{split}.parquet', params)
display(reports)

In [ ]:
metrics = train_reranker(candidate_dir / 'train.parquet', candidate_dir / 'valid.parquet', candidate_dir / 'test.parquet', reranker_dir, params)
display(metrics)

In [ ]:
# Chạy sau khi toàn bộ artifact tồn tại; đổi RUN_ID nếu cần chạy một thí nghiệm mới.
from datetime import datetime, timezone
from datn.experiments.coveo_checkpoint import checkpoint_coveo_run
RUN_ID = os.getenv('COVEO_RUN_ID', datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
bundle = checkpoint_coveo_run(REPO, REPO / 'checkpoints' / f'coveo_{RUN_ID}')
print('immutable checkpoint:', bundle)

Báo cáo đồng thời `candidate_recall`, `ConditionalHR@K` và HR end-to-end. Không force-add target vào candidates.